# Предшественники ФП/СФП за 365 дней: 2 подхода

Тетрадка считает TOP-5 предшественников ФП/СФП по сегментам (`МкБ`, `МСБ`, `ККБ`) для двух наборов целевых факторов:

1. `lines_1_3_only` — только 1-я и 3-я строки в сегменте (без строк в кавычках).
2. `all_lines` — все строки в сегменте (включая строки в кавычках).

Окно предшественников: **365 дней до даты целевого выявления** (строго раньше даты целевого события).

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

LOOKBACK_DAYS = 365
TOP_N = 5

# БАНКОВЫЕ пути (как в референсе analysis_default_logic_eda.ipynb).
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

# Период, в котором ищем целевые события.
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

# Для сопоставления наименований из справочника.
SEGMENT_TO_ZO = {
    "МкБ": {"ДМ (микро)"},
    "МСБ": {"ДМСБ (малый)", "ДМСБ (средний)"},
    "ККБ": {"ДКБ"},
}

# Наборы целевых факторов по запросу.
TARGETS = {
    "lines_1_3_only": {
        "МкБ": {
            "ФП": ["13", "13У", "15.1.2"],
            "СФП": ["15.2У", "15.2.1У", "15.2"],
        },
        "МСБ": {
            "ФП": ["13", "15.1.3"],
            "СФП": ["15.2"],
        },
        "ККБ": {
            "ФП": ["13", "13.1", "15.1.3.1"],
            "СФП": ["15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"],
        },
    },
    "all_lines": {
        "МкБ": {
            "ФП": ["13", "13У", "15.1У", "15.1.1У", "15.1.1", "15.1.2"],
            "СФП": ["15.2У", "15.2.1У", "15.2"],
        },
        "МСБ": {
            "ФП": ["13", "15.1.2", "15.1.3"],
            "СФП": ["15.2"],
        },
        "ККБ": {
            "ФП": ["13", "13.1", "15.1.2 (II)", "15.1.2 (III)", "15.1.2.1", "15.1.3.1"],
            "СФП": ["15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"],
        },
    },
}

_LAT2CYR = str.maketrans({
    "A": "А", "B": "В", "C": "С", "E": "Е", "H": "Н", "K": "К", "M": "М", "O": "О", "P": "Р", "T": "Т", "X": "Х", "Y": "У",
    "a": "а", "c": "с", "e": "е", "o": "о", "p": "р", "x": "х", "y": "у",
})


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def normalize_code(value):
    if pd.isna(value):
        return None
    s = str(value)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = s.replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s).strip().upper()
    s = re.sub(r"\s*\(\s*", " (", s)
    s = re.sub(r"\s*\)\s*", ")", s)
    return s


def normalize_type(value):
    if pd.isna(value):
        return None
    s = normalize_code(value)
    if s is None:
        return None
    if "SFP" in s or "СФП" in s:
        return "СФП"
    if re.search(r"(^|[^A-ZА-Я])FP([^A-ZА-Я]|$)", s) or "ФП" in s:
        return "ФП"
    return s


print("Конфигурация загружена.")
print(f"CRM_FILE: {CRM_FILE}")
print(f"REF_FILE: {REF_FILE}")
print(f"Период целей: {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}")
print(f"Окно предшественников: {LOOKBACK_DAYS} дней")

In [ ]:
crm_path = Path(CRM_FILE)
ref_path = Path(REF_FILE)

print("CRM:", crm_path)
print("REF:", ref_path)

crm_raw = pd.read_csv(crm_path, dtype=str, low_memory=False)
crm_raw.columns = crm_raw.columns.str.strip()

required_cols = ["X_INN", "IDENTIFICATION_DATE", "NUMBER_FP_SFP", "TYPE_FP", "X_AREA_RESP"]
missing = [c for c in required_cols if c not in crm_raw.columns]
if missing:
    raise KeyError(f"В CRM нет обязательных колонок: {missing}")

base = crm_raw.copy()
if "VAL" in base.columns:
    base = base[base["VAL"].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

base["inn"] = base["X_INN"].apply(normalize_inn)
base["dt"] = pd.to_datetime(base["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
base["segment"] = base["X_AREA_RESP"].astype(str).str.strip().map(SEGMENT_MAP)
base["fp_num"] = base["NUMBER_FP_SFP"].apply(normalize_code)
base["fp_type"] = base["TYPE_FP"].apply(normalize_type)

history_from = DATE_FROM - pd.Timedelta(days=LOOKBACK_DAYS)
base = base[(base["dt"] >= history_from) & (base["dt"] <= DATE_TO)].copy()
base = base[base["segment"].notna()].copy()
base = base.dropna(subset=["inn", "dt", "fp_num", "fp_type"]).copy()
base = base.drop_duplicates(subset=["inn", "segment", "fp_num", "fp_type", "dt"]).copy()

print(f"\nБаза после фильтров: {len(base):,} строк")
print(f"ИНН: {base['inn'].nunique():,}")
print(f"Период фактов: {base['dt'].min():%d.%m.%Y} - {base['dt'].max():%d.%m.%Y}")
if base["dt"].min() > history_from:
    print("⚠ История начинается позже нужного окна. Для корректного расчета по ранним целям нужен более ранний горизонт данных.")


def load_ref(ref_file: Path):
    candidates = [
        {"sep": "\t", "encoding": "utf-16"},
        {"sep": ";"},
        {"sep": ","},
    ]
    for params in candidates:
        try:
            df = pd.read_csv(ref_file, dtype=str, low_memory=False, **params)
            if len(df.columns) > 1:
                return df
        except Exception:
            continue
    return None


name_map = {}
ref = load_ref(ref_path)
if ref is None:
    raise FileNotFoundError(f"Не удалось прочитать ref_book.csv по пути: {ref_path}")

ref.columns = ref.columns.str.strip()
needed = {"№", "Наименование", "ЗО для ФП/СФП", "ФП", "СФП"}
if not needed.issubset(set(ref.columns)):
    raise KeyError("В ref_book.csv не хватает колонок: №, Наименование, ЗО для ФП/СФП, ФП, СФП")

ref = ref.copy()
ref["code"] = ref["№"].apply(normalize_code)
ref["zo"] = ref["ЗО для ФП/СФП"].astype(str).str.strip()
ref["name"] = ref["Наименование"].astype(str).str.strip()

fp_rows = ref[ref["ФП"].astype(str).str.upper().eq("Y")][["code", "zo", "name"]].copy()
fp_rows["type"] = "ФП"
sfp_rows = ref[ref["СФП"].astype(str).str.upper().eq("Y")][["code", "zo", "name"]].copy()
sfp_rows["type"] = "СФП"
ref_long = pd.concat([fp_rows, sfp_rows], ignore_index=True)

temp = {}
for row in ref_long.itertuples(index=False):
    for seg, zo_set in SEGMENT_TO_ZO.items():
        if row.zo in zo_set:
            key = (seg, row.type, row.code)
            temp.setdefault(key, set()).add(row.name)

name_map = {k: " / ".join(sorted(v)) for k, v in temp.items()}
print(f"Сопоставление наименований собрано: {len(name_map):,} ключей")

In [ ]:
def build_target_table(target_cfg):
    rows = []
    for seg, type_dict in target_cfg.items():
        for target_type, code_list in type_dict.items():
            for code in code_list:
                rows.append({
                    "segment": seg,
                    "target_type": target_type,
                    "target_code": normalize_code(code),
                })
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def compute_predecessors(base_df, target_cfg, approach_name):
    target_tbl = build_target_table(target_cfg)
    target_keys = set(zip(target_tbl["segment"], target_tbl["target_type"], target_tbl["target_code"]))

    work = base_df.copy()
    work["event_key"] = list(zip(work["segment"], work["fp_type"], work["fp_num"]))

    target_events = work[
        work["dt"].between(DATE_FROM, DATE_TO)
        & work["event_key"].isin(target_keys)
    ][["inn", "segment", "dt", "fp_type", "fp_num"]].copy()
    target_events = target_events.reset_index(drop=True)
    target_events["target_event_id"] = target_events.index.astype(int)

    target_coverage = (
        target_tbl.merge(
            target_events.groupby(["segment", "fp_type", "fp_num"], as_index=False)
            .size()
            .rename(columns={"fp_type": "target_type", "fp_num": "target_code", "size": "events_found"}),
            on=["segment", "target_type", "target_code"],
            how="left",
        )
        .fillna({"events_found": 0})
    )
    target_coverage["events_found"] = target_coverage["events_found"].astype(int)
    target_coverage.insert(0, "approach", approach_name)

    if len(target_events) == 0:
        empty_top = pd.DataFrame(
            columns=["approach", "segment", "rank_in_segment", "pred_type", "pred_code", "pred_name", "target_cases", "predecessor_events", "unique_inn"]
        )
        empty_summary = pd.DataFrame(
            [{"approach": approach_name, "segment": s, "target_events": 0, "target_inn": 0, "target_events_with_predecessor": 0} for s in ["МкБ", "МСБ", "ККБ"]]
        )
        return {
            "target_events": target_events,
            "target_coverage": target_coverage,
            "top5": empty_top,
            "summary": empty_summary,
        }

    pred_pool = work[["inn", "segment", "dt", "fp_type", "fp_num"]].copy()

    merged = (
        target_events[["target_event_id", "inn", "segment", "dt", "fp_type", "fp_num"]]
        .rename(columns={"dt": "target_dt", "fp_type": "target_type", "fp_num": "target_code"})
        .merge(pred_pool, on=["inn", "segment"], how="left", suffixes=("", "_pred"))
        .rename(columns={"dt": "pred_dt", "fp_type": "pred_type", "fp_num": "pred_code"})
    )

    merged = merged[
        (merged["pred_dt"] < merged["target_dt"])
        & (merged["pred_dt"] >= (merged["target_dt"] - pd.Timedelta(days=LOOKBACK_DAYS)))
    ].copy()

    merged_unique = merged.drop_duplicates(subset=["target_event_id", "pred_type", "pred_code"]).copy()

    if len(merged_unique) == 0:
        top5 = pd.DataFrame(
            columns=["approach", "segment", "rank_in_segment", "pred_type", "pred_code", "pred_name", "target_cases", "predecessor_events", "unique_inn"]
        )
    else:
        agg = (
            merged_unique.groupby(["segment", "pred_type", "pred_code"], as_index=False)
            .agg(
                target_cases=("target_event_id", "nunique"),
                predecessor_events=("pred_dt", "count"),
                unique_inn=("inn", "nunique"),
            )
            .sort_values(
                ["segment", "target_cases", "predecessor_events", "unique_inn", "pred_type", "pred_code"],
                ascending=[True, False, False, False, True, True],
            )
        )
        agg["rank_in_segment"] = agg.groupby("segment").cumcount() + 1
        top5 = agg[agg["rank_in_segment"] <= TOP_N].copy()

    top5.insert(0, "approach", approach_name)
    if len(top5):
        top5["pred_name"] = top5.apply(
            lambda r: name_map.get((r["segment"], r["pred_type"], r["pred_code"]), ""),
            axis=1,
        )
    else:
        top5["pred_name"] = ""

    summary = (
        target_events.groupby("segment", as_index=False)
        .agg(target_events=("target_event_id", "count"), target_inn=("inn", "nunique"))
    )
    if len(merged_unique) > 0:
        with_pred = (
            merged_unique.groupby("segment", as_index=False)
            .agg(target_events_with_predecessor=("target_event_id", "nunique"))
        )
        summary = summary.merge(with_pred, on="segment", how="left")
    else:
        summary["target_events_with_predecessor"] = 0
    summary["target_events_with_predecessor"] = summary["target_events_with_predecessor"].fillna(0).astype(int)

    for seg in ["МкБ", "МСБ", "ККБ"]:
        if seg not in set(summary["segment"]):
            summary = pd.concat(
                [summary, pd.DataFrame([{"segment": seg, "target_events": 0, "target_inn": 0, "target_events_with_predecessor": 0}])],
                ignore_index=True,
            )

    summary = summary.sort_values("segment")
    summary.insert(0, "approach", approach_name)

    return {
        "target_events": target_events,
        "target_coverage": target_coverage,
        "top5": top5,
        "summary": summary,
    }


results = {name: compute_predecessors(base, cfg, name) for name, cfg in TARGETS.items()}

summary_all = pd.concat([results[name]["summary"] for name in TARGETS], ignore_index=True)
coverage_all = pd.concat([results[name]["target_coverage"] for name in TARGETS], ignore_index=True)
top5_all = pd.concat([results[name]["top5"] for name in TARGETS], ignore_index=True)
target_events_all = pd.concat(
    [results[name]["target_events"].assign(approach=name) for name in TARGETS],
    ignore_index=True,
)

coverage_all["is_found"] = coverage_all["events_found"] > 0
missing_targets = coverage_all[~coverage_all["is_found"]].copy()
found_targets = coverage_all[coverage_all["is_found"]].copy()

print("Сводка по целевым событиям:")
display(summary_all.sort_values(["approach", "segment"]))

print("Покрытие целевых кодов (сколько событий найдено для каждого кода):")
display(coverage_all.sort_values(["approach", "segment", "target_type", "target_code"]))

print("\nДиагностика покрытия кодов:")
print(f"  Найдено кодов с событиями: {len(found_targets):,}")
print(f"  Не найдено кодов в периоде: {len(missing_targets):,}")
if len(missing_targets):
    display(
        missing_targets[["approach", "segment", "target_type", "target_code", "events_found"]]
        .sort_values(["approach", "segment", "target_type", "target_code"])
    )

for approach in TARGETS:
    print("=" * 95)
    print(f"TOP-{TOP_N} предшественников | подход: {approach}")
    print("=" * 95)
    cur = top5_all[top5_all["approach"] == approach].copy()
    if len(cur) == 0:
        print("Нет найденных предшественников для выбранных целей.")
        continue
    for seg in ["МкБ", "МСБ", "ККБ"]:
        print(f"\nСегмент: {seg}")
        sub = cur[cur["segment"] == seg].copy().sort_values("rank_in_segment")
        if len(sub) == 0:
            print("  Нет данных")
        else:
            display(
                sub[[
                    "rank_in_segment",
                    "pred_type",
                    "pred_code",
                    "pred_name",
                    "target_cases",
                    "predecessor_events",
                    "unique_inn",
                ]]
            )

In [ ]:
OUT_XLSX = Path(DATA_DIR) / "predecessors_top5_two_approaches.xlsx"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    summary_all.sort_values(["approach", "segment"]).to_excel(writer, sheet_name="summary", index=False)
    coverage_all.sort_values(["approach", "segment", "target_type", "target_code"]).to_excel(writer, sheet_name="target_coverage", index=False)
    top5_all.sort_values(["approach", "segment", "rank_in_segment"]).to_excel(writer, sheet_name="top5_predecessors", index=False)
    target_events_all.sort_values(["approach", "segment", "dt"]).to_excel(writer, sheet_name="target_events", index=False)
    missing_targets.sort_values(["approach", "segment", "target_type", "target_code"]).to_excel(writer, sheet_name="missing_targets", index=False)

print(f"Сохранено: {OUT_XLSX}")